# 04 · Capa oro: tabla para el modelo

**Qué hace:** une las tablas de plata en `gold_contratos`: una fila por contrato con la
**etiqueta** y las **variables conocidas al firmar**.

**¿Por qué desnormalizar si plata está en 3FN?** Plata está normalizada para *guardar*
bien (sin redundancia, con integridad). Oro se desnormaliza a propósito para *analizar*:
los descriptivos y el modelo necesitan todo en una fila, y repetir 6 joins en cada consulta
sería lento y propenso a errores. Normalizar para almacenar, desnormalizar para analizar.

- **Etiqueta:** `etiqueta = 1` si `dias_adicionados > 0` (hubo prórroga), `0` si es 0.
  Si `dias_adicionados` está vacío, no sabemos qué pasó y la fila se excluye.
- **Variables:** tipo de contrato, modalidad y justificación (de `dim_proceso`), entidad
  (orden, sector, rama, departamento), proveedor (tipo de documento, grupo), pyme, categoría
  UNSPSC (familia y segmento, de `dim_familia_unspsc`), valor (recortado y en logaritmo),
  calendario de la firma y distancia firma-inicio.
- **Excluido por fuga:** estado, pagos, saldos, fechas de liquidación (nunca se unen).
- **Plazo (fin - inicio) y duración:** se guardan aparte porque NO sabemos si ya incluyen
  los días añadidos. El notebook 06 compara el modelo con y sin ellas.
- **Años 2025-2026:** quedan en la tabla con `etiqueta_completa = false` y no se usan
  para entrenar (aún pueden recibir prórrogas).

## Configuración: importar `src/secop` y crear la sesión de Spark

1. Buscamos la carpeta del repositorio (la que contiene `src/secop`) subiendo desde la
   carpeta del notebook y añadimos `src` a `sys.path`, la lista de carpetas donde Python
   busca módulos. Así `import secop` funciona sin instalar nada.
2. `entorno.crear_spark()` crea una sesión de **Spark en modo local** con Delta Lake: el
   mismo motor que en un clúster, usando los núcleos del Codespace.

Si cambias código en `src/`, reinicia el kernel (botón *Restart*) para que Python lo
vuelva a cargar.

In [ ]:
import sys
from pathlib import Path


def buscar_raiz_repo() -> Path:
    """Devuelve la carpeta raíz del repo (la que contiene src/secop)."""
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / "src" / "secop").is_dir():
            return carpeta
    raise RuntimeError("No encontré src/secop: abre el notebook desde la carpeta del repositorio.")


RAIZ_REPO = buscar_raiz_repo()
if str(RAIZ_REPO / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ_REPO / "src"))

import secop
from secop import config, entorno

spark = entorno.crear_spark()
print("secop", secop.__version__, "| Spark", spark.version, "| datos en", config.raiz_datos())

## 1. Construir oro
Las dimensiones pequeñas (entidad, categoría, familia) se unen con **broadcast join**:
Spark envía una copia completa a cada tarea y así evita redistribuir (*shuffle*) la tabla
grande de contratos.

In [ ]:
from pyspark.sql import functions as F

from secop.esquema import COLUMNAS_POSTERIORES_FIRMA
from secop.oro import construir_oro
from secop.validacion import exigir_sin_personales


def leer(nombre):
    return entorno.leer_tabla(spark, nombre)


oro = construir_oro(
    contrato=leer(config.TABLA_CONTRATO),
    dim_entidad=leer(config.TABLA_DIM_ENTIDAD),
    dim_proveedor=leer(config.TABLA_DIM_PROVEEDOR),
    dim_categoria=leer(config.TABLA_DIM_CATEGORIA),
    dim_familia=leer(config.TABLA_DIM_FAMILIA),
    dim_proceso=leer(config.TABLA_DIM_PROCESO),
    ejecucion=leer(config.TABLA_EJECUCION),
)

# Verificaciones ANTES de escribir: sin datos personales y sin columnas posteriores a la firma.
exigir_sin_personales(oro.columns)
fuga = [c for c in oro.columns if c in COLUMNAS_POSTERIORES_FIRMA]
assert not fuga, f"Columnas con fuga de información en oro: {fuga}"

entorno.escribir_tabla(oro, config.TABLA_ORO)
print("escrita:", config.TABLA_ORO)

## 2. Verificación
Filas y tasa de prórroga por año de firma.
Referencia de la exploración previa para 2019: **141.257 filas y 10,99 % con etiqueta 1**.
Si el resultado difiere, revisar los filtros de `construir_oro` (valor > 0, fecha de firma
y etiqueta conocidas) y anotar la diferencia en `docs/informe.md`.

In [ ]:
gold = entorno.leer_tabla(spark, config.TABLA_ORO)
entorno.mostrar(
    gold.groupBy("anio_firma", "etiqueta_completa")
    .agg(
        F.count("*").alias("contratos"),
        F.sum("etiqueta").alias("prorrogados"),
        F.round(F.avg("etiqueta") * 100, 2).alias("tasa_pct"),
    )
    .orderBy("anio_firma")
)

## 3. Completitud de las variables
Porcentaje de vacíos en las variables numéricas (las categóricas nunca son nulas: un vacío
se convierte en la categoría "No Definido").

In [ ]:
numericas = config.VARIABLES_NUMERICAS + config.VARIABLES_PLAZO
entorno.mostrar(
    gold.select(
        [F.round(F.avg(F.col(c).isNull().cast("int")) * 100, 2).alias(c) for c in numericas]
    )
)